# Powerlifting Bronze Layer

Downloads the OpenPowerlifting Kaggle dataset and appends each snapshot into the `workspace.powerlifting_bronze` schema with `_ingested_at` load provenance and Delta Change Data Feed enabled for incremental downstream processing.

In [0]:
import pandas as pd
import os
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dansbecker/powerlifting-database")

print("Path to dataset files:", path)

dataset_path = "/home/spark-6c0e28bd-0f58-426e-9d18-ab/.cache/kagglehub/datasets/binayas/powerlifting-dataset/versions/12"

# List all files in the dataset (recursively)
for root, dirs, files in os.walk(dataset_path):
    for f in sorted(files):
        rel = os.path.relpath(os.path.join(root, f), dataset_path)
        size_mb = os.path.getsize(os.path.join(root, f)) / 1024 / 1024
        print(f"{rel}  ({size_mb:.2f} MB)")

100%|██████████| 8.85M/8.85M [00:00<00:00, 62.4MB/s]

Extracting files...


Path to dataset files: /home/spark-f65104e3-bde5-4678-9973-86/.cache/kagglehub/datasets/dansbecker/powerlifting-database/versions/1


In [0]:
import os
import pandas as pd
from pyspark.sql.functions import current_timestamp

# Load CSV files from the downloaded dataset into a dictionary of Spark DataFrames
# Using pandas to read local files (Spark can't access driver filesystem on Serverless)
dfs = {}
for f in os.listdir(path):
    if f.endswith(".csv"):
        table_name = f.replace(".csv", "")
        pdf = pd.read_csv(os.path.join(path, f))
        dfs[table_name] = spark.createDataFrame(pdf)
        print(f"Loaded {table_name} ({dfs[table_name].count()} rows)")

# Create the powerlifting_bronze schema
spark.sql("CREATE SCHEMA IF NOT EXISTS powerlifting_bronze")

# Save every table into the powerlifting_bronze schema
# Add _ingested_at timestamp for load provenance and enable Delta CDF so future incremental runs can detect what changed (inserts/updates/deletes).
# Write mode is APPEND - each run adds the latest Kaggle snapshot as a new batch.
# Duplicates are expected (same rows re-loaded); silver layer deduplicates via MERGE.
for name, df in dfs.items():
    df.withColumn("_ingested_at", current_timestamp()) \
     .write.format("delta") \
     .option("mergeSchema", "true") \
     .mode("append") \
     .saveAsTable(f"powerlifting_bronze.{name}")
    print(f"Appended to powerlifting_bronze.{name} (with _ingested_at)")

# Enable Delta Change Data Feed (CDF) on all bronze tables via ALTER TABLE
for name in dfs.keys():
    spark.sql(f"ALTER TABLE powerlifting_bronze.{name} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
    print(f"CDF enabled on powerlifting_bronze.{name}")

# Verify
display(spark.sql("SHOW TABLES IN powerlifting_bronze"))

# Quick check: show CDF + ingestion column on one table
display(spark.sql("DESCRIBE TABLE EXTENDED powerlifting_bronze.openpowerlifting").filter("col_name IN ('_ingested_at', 'delta.enableChangeDataFeed')"))

Loaded openpowerlifting (386414 rows)
Loaded meets (8482 rows)
Appended to powerlifting_bronze.openpowerlifting (with _ingested_at)
Appended to powerlifting_bronze.meets (with _ingested_at)
CDF enabled on powerlifting_bronze.openpowerlifting
CDF enabled on powerlifting_bronze.meets


database,tableName,isTemporary
powerlifting_bronze,meets,false
powerlifting_bronze,openpowerlifting,false


col_name,data_type,comment
_ingested_at,timestamp,null
